# Classifying by One Variable

Data scientists often need to classify individuals into groups according to shared features, and then identify some characteristics of the groups. For example, in the example using Galton's data on heights, we saw that it was useful to classify families according to the parents' midparent heights, and then find the average height of the children in each group.

This section is about classifying individuals into categories that are not numerical. We begin by recalling the basic use of pandas `groupby` operations.

In [1]:
import pandas as pd
path_data = "../../../assets/data/"

import matplotlib
matplotlib.use("Agg")
%matplotlib inline
import matplotlib.pyplot as plt
plt.style.use("fivethirtyeight")
import numpy as np

## Counting the Number in Each Category

The pandas `groupby` method can be used to count the number of rows for each category in a column. The result contains one row per unique value in the grouped column.

Here is a small data set on ice cream cones. We can use `groupby` to list the distinct flavors and provide the count of each flavor.

In [2]:
cones = pd.DataFrame({"Flavor":["strawberry","chocolate","chocolate","strawberry","chocolate"],"Price":[3.55,4.75,6.55,5.25,5.25]})
cones

,Flavor,Price
0,strawberry,3.55
1,chocolate,4.75
2,chocolate,6.55
3,strawberry,5.25
4,chocolate,5.25


In [3]:
cones.groupby("Flavor").size().reset_index(name="count")

,Flavor,count
0,chocolate,3
1,strawberry,2


There are two distinct categories, chocolate and strawberry. The `groupby(...).size()` operation creates a table of counts in each category. The `count` column contains the number of rows in each category.

Notice that this can all be worked out from just the `Flavor` column. The `Price` column has not been used.

But what if we wanted the total price of the cones of each different flavor? That is where aggregation functions come in.

## Finding a Characteristic of Each Category

After grouping by a category, we can use an aggregation function to summarize values in the other columns.

In [4]:
cones.groupby("Flavor", as_index=False)["Price"].sum()

,Flavor,Price
0,chocolate,16.55
1,strawberry,8.80


In [5]:
cones.loc[cones["Flavor"]=="chocolate","Price"]

1    4.75
2    6.55
4    5.25
Name: Price, dtype: float64

In [6]:
cones.loc[cones["Flavor"]=="chocolate","Price"].sum()

np.float64(16.55)

In [7]:
cones_choc=cones.loc[cones["Flavor"]=="chocolate","Price"]
cones_strawb=cones.loc[cones["Flavor"]=="strawberry","Price"]
grouped_cones=pd.DataFrame({"Flavor":["chocolate","strawberry"],"Array of All the Prices":[list(cones_choc),list(cones_strawb)]})
price_totals=grouped_cones.assign(**{"Sum of the Array":[cones_choc.sum(),cones_strawb.sum()]})
price_totals

,Flavor,Array of All the Prices,Sum of the Array
0,chocolate,"[4.75, 6.55, 5.25]",16.55
1,strawberry,"[3.55, 5.25]",8.80


In [8]:
cones.groupby("Flavor", as_index=False)["Price"].max()

,Flavor,Price
0,chocolate,6.55
1,strawberry,5.25


In [9]:
price_maxes=grouped_cones.assign(**{"Max of the Array":[cones_choc.max(),cones_strawb.max()]})
price_maxes

,Flavor,Array of All the Prices,Max of the Array
0,chocolate,"[4.75, 6.55, 5.25]",6.55
1,strawberry,"[3.55, 5.25]",5.25


In [10]:
lengths=grouped_cones.assign(**{"Length of the Array":[len(cones_choc),len(cones_strawb)]})
lengths

,Flavor,Array of All the Prices,Length of the Array
0,chocolate,"[4.75, 6.55, 5.25]",3
1,strawberry,"[3.55, 5.25]",2


## Example: NBA Salaries

In [11]:
nba=pd.read_csv(path_data+"nba_salaries.csv")
nba=nba.rename(columns={"'15-'16 SALARY":"SALARY"})
nba

,PLAYER,POSITION,TEAM,SALARY
0,Paul Millsap,PF,Atlanta Hawks,18.671659
1,Al Horford,C,Atlanta Hawks,12.000000
2,Tiago Splitter,C,Atlanta Hawks,9.756250
3,Jeff Teague,PG,Atlanta Hawks,8.000000
4,Kyle Korver,SG,Atlanta Hawks,5.746479
...,...,...,...,...
412,Gary Neal,PG,Washington Wizards,2.139000
413,DeJuan Blair,C,Washington Wizards,2.000000
414,Kelly Oubre Jr.,SF,Washington Wizards,1.920240
415,Garrett Temple,SG,Washington Wizards,1.100602


**1.** How much money did each team pay for its players' salaries?

In [12]:
teams_and_money=nba[["TEAM","SALARY"]]
teams_and_money.groupby("TEAM",as_index=False)["SALARY"].sum()

,TEAM,SALARY
0,Atlanta Hawks,69.573103
1,Boston Celtics,50.285499
2,Brooklyn Nets,57.306976
3,Charlotte Hornets,84.102397
4,Chicago Bulls,78.820890
5,Cleveland Cavaliers,102.312412
6,Dallas Mavericks,65.762559
7,Denver Nuggets,62.429404
8,Detroit Pistons,42.211760
9,Golden State Warriors,94.085137


**2.** How many NBA players were there in each of the five positions?

In [13]:
nba.groupby("POSITION").size().reset_index(name="count")

,POSITION,count
0,C,69
1,PF,85
2,PG,85
3,SF,82
4,SG,96


**3.** What was the average salary of the players at each of the five positions?

In [14]:
positions_and_money=nba[["POSITION","SALARY"]]
positions_and_money.groupby("POSITION",as_index=False)["SALARY"].mean()

,POSITION,SALARY
0,C,6.082913
1,PF,4.951344
2,PG,5.165487
3,SF,5.532675
4,SG,3.988195


In [15]:
nba.groupby("POSITION").mean(numeric_only=True).reset_index()

,POSITION,SALARY
0,C,6.082913
1,PF,4.951344
2,PG,5.165487
3,SF,5.532675
4,SG,3.988195
